In [5]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy.ma as ma

# --- Inputs ---
raster_path = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_wo_VLM_stat.tif"
boundary_path = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

# Flood definition threshold (meters)
FLOOD_THRESH = 0.1

# --- Read boundary (should be EPSG:32614 for correct area calc in meters) ---
boundary = gpd.read_file(boundary_path)
if boundary.crs is None:
    raise ValueError("Boundary CRS is None. Set the CRS (EPSG:32614) before proceeding.")

# --- Read raster and clip to boundary ---
with rasterio.open(raster_path) as src:
    raster_crs = src.crs
    if raster_crs is None:
        raise ValueError("Raster CRS is None. It must be defined (expected EPSG:32614).")
    boundary_proj = boundary.to_crs(raster_crs)

    out_img, out_transform = mask(src, boundary_proj.geometry, crop=True)
    depth = out_img[0].astype("float64")
    nodata = src.nodata

# --- Clean data ---
if nodata is not None:
    depth = np.where(depth == nodata, np.nan, depth)
depth = np.where(depth < 0, np.nan, depth)  # negative values → NaN
depth_ma = ma.masked_invalid(depth)

# --- Pixel area (m²) ---
px_w = abs(out_transform.a)
px_h = abs(out_transform.e)
cell_area_m2 = px_w * px_h

# --- Land-part area (km²) ---
land_area_m2 = boundary_proj.geometry.area.sum()
land_area_km2 = land_area_m2 / 1e6

# --- Flood mask and stats ---
flood_mask = (depth_ma >= FLOOD_THRESH)

flooded_cells = flood_mask.filled(False).sum()
flood_area_km2 = (flooded_cells * cell_area_m2) / 1e6

percent_land_flooded = (flood_area_km2 / land_area_km2) * 100 if land_area_km2 > 0 else np.nan

flooded_depth_values = depth_ma[flood_mask]
if flooded_depth_values.count() == 0:
    max_depth = np.nan
    median_depth = np.nan
    mean_depth = np.nan
else:
    max_depth = float(flooded_depth_values.max())
    median_depth = float(ma.median(flooded_depth_values))
    mean_depth = float(flooded_depth_values.mean())

# --- Print results ---
print("=== Flood Summary (threshold >= {:.2f} m) ===".format(FLOOD_THRESH))
print("Land-part area:        {:.3f} km²".format(land_area_km2))
print("Flooded area:          {:.3f} km²".format(flood_area_km2))
print("Percent land flooded:  {:.2f} %".format(percent_land_flooded))
print("Max depth (flooded):   {:.3f} m".format(max_depth))
print("Median depth (flooded):{:.3f} m".format(median_depth))
print("Mean depth (flooded):  {:.3f} m".format(mean_depth))


=== Flood Summary (threshold >= 0.10 m) ===
Land-part area:        13918.672 km²
Flooded area:          13720.840 km²
Percent land flooded:  98.58 %
Max depth (flooded):   19.903 m
Median depth (flooded):4.185 m
Mean depth (flooded):  4.273 m


In [6]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy.ma as ma

# --- Inputs ---
raster_path = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"
boundary_path = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

# Flood definition threshold (meters)
FLOOD_THRESH = 0.1

# --- Read boundary (should be EPSG:32614 for area in meters) ---
boundary = gpd.read_file(boundary_path)
if boundary.crs is None:
    raise ValueError("Boundary CRS is None. Set the CRS (EPSG:32614) before proceeding.")

# --- Read raster and clip to boundary ---
with rasterio.open(raster_path) as src:
    raster_crs = src.crs
    if raster_crs is None:
        raise ValueError("Raster CRS is None. It must be defined (expected EPSG:32614).")
    boundary_proj = boundary.to_crs(raster_crs)

    out_img, out_transform = mask(src, boundary_proj.geometry, crop=True)
    depth = out_img[0].astype("float64")
    nodata = src.nodata

# --- Clean data: NoData -> NaN, restrict to [0, 5] m ---
if nodata is not None:
    depth = np.where(depth == nodata, np.nan, depth)

# keep only 0–5 m, discard <0 and >5
depth = np.where((depth < 0) | (depth > 5), np.nan, depth)

# Mask invalid for stats
depth_ma = ma.masked_invalid(depth)

# --- Pixel area (m²) ---
px_w = abs(out_transform.a)
px_h = abs(out_transform.e)
cell_area_m2 = px_w * px_h

# --- Land-part area (km²) ---
land_area_m2 = boundary_proj.geometry.area.sum()
land_area_km2 = land_area_m2 / 1e6

# --- Flood mask and stats (using the 0–5 m filtered raster) ---
flood_mask = (depth_ma >= FLOOD_THRESH)

flooded_cells = flood_mask.filled(False).sum()
flood_area_km2 = (flooded_cells * cell_area_m2) / 1e6

percent_land_flooded = (flood_area_km2 / land_area_km2) * 100 if land_area_km2 > 0 else np.nan

flooded_depth_values = depth_ma[flood_mask]
if flooded_depth_values.count() == 0:
    max_depth = np.nan
    median_depth = np.nan
    mean_depth = np.nan
else:
    max_depth = float(flooded_depth_values.max())
    median_depth = float(ma.median(flooded_depth_values))
    mean_depth = float(flooded_depth_values.mean())

# --- Print results ---
print("=== Flood Summary (depth limited to 0–5 m, threshold >= {:.2f} m) ===".format(FLOOD_THRESH))
print("Land-part area:        {:.3f} km²".format(land_area_km2))
print("Flooded area:          {:.3f} km²".format(flood_area_km2))
print("Percent land flooded:  {:.2f} %".format(percent_land_flooded))
print("Max depth (flooded):   {:.3f} m".format(max_depth))
print("Median depth (flooded):{:.3f} m".format(median_depth))
print("Mean depth (flooded):  {:.3f} m".format(mean_depth))


=== Flood Summary (depth limited to 0–5 m, threshold >= 0.10 m) ===
Land-part area:        13918.672 km²
Flooded area:          8442.120 km²
Percent land flooded:  60.65 %
Max depth (flooded):   5.000 m
Median depth (flooded):3.160 m
Mean depth (flooded):  3.023 m


In [10]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy.ma as ma

# --- Inputs ---
raster_path = r"D:\Phd Research\Final_Raster\inundation only due ot SLR+Q_v1_stat.tif"
boundary_path = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

# Flood definition threshold (meters)
FLOOD_THRESH = 0.1

# --- Read boundary (should be EPSG:32614 for correct area calc in meters) ---
boundary = gpd.read_file(boundary_path)
if boundary.crs is None:
    raise ValueError("Boundary CRS is None. Set the CRS (EPSG:32614) before proceeding.")

# --- Read raster and clip to boundary ---
with rasterio.open(raster_path) as src:
    raster_crs = src.crs
    if raster_crs is None:
        raise ValueError("Raster CRS is None. It must be defined (expected EPSG:32614).")
    boundary_proj = boundary.to_crs(raster_crs)

    out_img, out_transform = mask(src, boundary_proj.geometry, crop=True)
    depth = out_img[0].astype("float64")
    nodata = src.nodata

# --- Clean data ---
if nodata is not None:
    depth = np.where(depth == nodata, np.nan, depth)
depth = np.where(depth < 0, np.nan, depth)  # negative values → NaN
depth_ma = ma.masked_invalid(depth)

# --- Pixel area (m²) ---
px_w = abs(out_transform.a)
px_h = abs(out_transform.e)
cell_area_m2 = px_w * px_h

# --- Land-part area (km²) ---
land_area_m2 = boundary_proj.geometry.area.sum()
land_area_km2 = land_area_m2 / 1e6

# --- Flood mask and stats ---
flood_mask = (depth_ma >= FLOOD_THRESH)

flooded_cells = int(flood_mask.filled(False).sum())
flood_area_km2 = (flooded_cells * cell_area_m2) / 1e6
percent_land_flooded = (flood_area_km2 / land_area_km2) * 100 if land_area_km2 > 0 else np.nan

flooded_depth_values = depth_ma[flood_mask]

if flooded_depth_values.count() == 0:
    max_depth = np.nan
    median_depth = np.nan
    mean_depth = np.nan
    p95_depth = np.nan
    p98_depth = np.nan
    n_top5 = 0
    n_top2 = 0
else:
    # Basic stats
    min_depth = float(flooded_depth_values.min())
    max_depth = float(flooded_depth_values.max())
    median_depth = float(ma.median(flooded_depth_values))
    mean_depth = float(flooded_depth_values.mean())

    # Convert masked array to plain 1D ndarray for percentiles
    vals = flooded_depth_values.compressed().astype(float)

    # Percentiles
    p95_depth = float(np.nanpercentile(vals, 95))
    p98_depth = float(np.nanpercentile(vals, 98))

    # Counts of deepest 5% and 2% cells (≥ percentile thresholds)
    n_total = vals.size
    n_top5 = int((vals >= p95_depth).sum())
    n_top2 = int((vals >= p98_depth).sum())

# --- Print results ---
print("=== Flood Summary (threshold >= {:.2f} m) ===".format(FLOOD_THRESH))
print("Land-part area:               {:.3f} km²".format(land_area_km2))
print("Flooded area:                 {:.3f} km²".format(flood_area_km2))
print("Percent land flooded:         {:.2f} %".format(percent_land_flooded))
print("Flooded cell count:           {:,}".format(flooded_cells))
print("Min depth (flooded):          {:.3f} m".format(min_depth))
print("Max depth (flooded):          {:.3f} m".format(max_depth))
print("Median depth (flooded):       {:.3f} m".format(median_depth))
print("Mean depth (flooded):         {:.3f} m".format(mean_depth))
print("95th percentile depth:        {:.3f} m".format(p95_depth))
print("98th percentile depth:        {:.3f} m".format(p98_depth))
print("Cells in deepest 5% (≥P95):   {:,}".format(n_top5))
print("Cells in deepest 2% (≥P98):   {:,}".format(n_top2))


=== Flood Summary (threshold >= 0.10 m) ===
Land-part area:               13918.672 km²
Flooded area:                 4858.720 km²
Percent land flooded:         34.91 %
Flooded cell count:           121,468
Min depth (flooded):          0.100 m
Max depth (flooded):          14.075 m
Median depth (flooded):       0.581 m
Mean depth (flooded):         1.054 m
95th percentile depth:        3.048 m
98th percentile depth:        4.472 m
Cells in deepest 5% (≥P95):   6,074
Cells in deepest 2% (≥P98):   2,430
